This notebook builds a dataset for human-vs-machine translation detection:

- `process_and_shuffle_sentences`: reads fixed number of lines from source text files, filters non-empty lines, collects and shuffles them, writes to output.
- Two data preparation runs:
    - from multiple MT outputs to `shuffled_output.txt`
    - from clean human corpus to `ht.txt`
- `create_labeled_dataset`: loads two sentence files, creates train/val/test splits with labels (0 for human, 1 for machine), shuffles each split, writes CSV files.
- Final call generates `train.csv`, `val.csv`, `test.csv` from prepared `ht.txt` and `shuffled_output.txt`.

In [ ]:
import random
import os

def process_and_shuffle_sentences(sources, num_sent, output_file):
    """
    Reads a specified number of sentences from multiple source files,
    shuffles them, and saves them to a new file.

    A line is considered a valid sentence only if it starts with a 
    capital letter and ends with a period (full-stop).

    Args:
        sources (list): A list of strings, where each string is a path to a source file.
        num_sent (int): The number of valid sentences to read from each source file.
        output_file (str): The path to the file where the shuffled sentences will be saved.
    """
    print(f"Starting to process {len(sources)} source file(s)...")
    
    # This list will hold all the valid sentences from all files
    all_sentences = []

    # Iterate over each source file path provided
    for file_path in sources:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                sentences_from_this_file = []
                # Read each line from the current file
                for line in f:
                    # Strip leading/trailing whitespace to properly check conditions
                    clean_line = line.strip()

                    # Check if the line is not empty and meets the criteria
                    if clean_line:
                        sentences_from_this_file.append(clean_line)
                        # Stop if we have found the desired number of sentences
                        if len(sentences_from_this_file) >= num_sent:
                            break
                
                print(f"-> Found {len(sentences_from_this_file)} valid sentences in '{file_path}'.")
                all_sentences.extend(sentences_from_this_file)

        except FileNotFoundError:
            print(f"Warning: The file '{file_path}' was not found. Skipping.")
        except Exception as e:
            print(f"An error occurred with file '{file_path}': {e}")

    # --- Shuffling and Saving ---
    print(f"\nTotal valid sentences collected: {len(all_sentences)}")
    
    if not all_sentences:
        print("No sentences were collected. The output file will be empty.")
        # Create an empty file
        open(output_file, 'w').close()
        return

    print("Shuffling sentences...")
    random.shuffle(all_sentences)

    print(f"Saving shuffled sentences to '{output_file}'...")
    try:
        with open(output_file, 'w', encoding='utf-8') as f_out:
            # Write each sentence on a new line
            for sentence in all_sentences:
                f_out.write(sentence + '\n')
        print("Process complete!")
    except Exception as e:
        print(f"Error: Could not write to output file '{output_file}'. Reason: {e}")

## Data Preparation Strategy

### For English:
- Remove English files from the first cell (MT sources)
- Add English file to the second cell (HT sources)

### For Spanish:
- Remove Spanish files from the first cell (MT sources)
- Add Spanish file to the second cell (HT sources)


In [ ]:
source_files = ["parallel_en.txt", "parallel_fr.txt", "parallel_de.txt", "parallel_es.txt"] # Set the path to the MT files
sentences_per_file = 20000 # Adjust this number based on how many sentences you want to read from each file, this is the number of sentence used for the generation of MT sentences
final_output_file = "shuffled_output.txt"

process_and_shuffle_sentences(
    sources=source_files, 
    num_sent=sentences_per_file, 
    output_file=final_output_file
)
    

In [ ]:
source_files = ["parallel_en.txt"] # Set the path to the HT files
sentences_per_file = 60000 # Adjust this number based on how many sentences you want to read from each file, this is the number of sentence used for HT sentences
final_output_file = "ht.txt"

process_and_shuffle_sentences(
    sources=source_files, 
    num_sent=sentences_per_file, 
    output_file=final_output_file
)

In [6]:
import csv

def create_labeled_dataset(
    file1_path, file2_path,
    train_num, val_num, test_num,
    train_csv_path, val_csv_path, test_csv_path
):
    # Read sentences from both files
    with open(file1_path, "r", encoding="utf-8") as f1, open(file2_path, "r", encoding="utf-8") as f2:
        sents1 = [line.strip() for line in f1 if line.strip()]
        sents2 = [line.strip() for line in f2 if line.strip()]

    # Training split
    train_data = []
    train_data.extend([{"sentence": sent, "label": 0} for sent in sents1[:train_num]])
    train_data.extend([{"sentence": sent, "label": 1} for sent in sents2[:train_num]])

    # Validation split
    val_data = []
    val_data.extend([{"sentence": sent, "label": 0} for sent in sents1[train_num:train_num+val_num]])
    val_data.extend([{"sentence": sent, "label": 1} for sent in sents2[train_num:train_num+val_num]])

    # Testing split
    test_data = []
    test_data.extend([{"sentence": sent, "label": 0} for sent in sents1[train_num+val_num:train_num+val_num+test_num]])
    test_data.extend([{"sentence": sent, "label": 1} for sent in sents2[train_num+val_num:train_num+val_num+test_num]])

    # Shuffle each split to mix labels
    import random
    random.shuffle(train_data)
    random.shuffle(val_data)
    random.shuffle(test_data)

    # Write train.csv
    with open(train_csv_path, "w", encoding="utf-8", newline="") as fcsv:
        writer = csv.DictWriter(fcsv, fieldnames=["sentence", "label"])
        writer.writeheader()
        for row in train_data:
            writer.writerow(row)

    # Write val.csv
    with open(val_csv_path, "w", encoding="utf-8", newline="") as fcsv:
        writer = csv.DictWriter(fcsv, fieldnames=["sentence", "label"])
        writer.writeheader()
        for row in val_data:
            writer.writerow(row)

    # Write test.csv
    with open(test_csv_path, "w", encoding="utf-8", newline="") as fcsv:
        writer = csv.DictWriter(fcsv, fieldnames=["sentence", "label"])
        writer.writeheader()
        for row in test_data:
            writer.writerow(row)



In [ ]:
# Train, val, test splits generation
ht = "" # Set the path to the HT sentences created
mt = "" # Set the path to the MT sentences created
train_num = 50000
val_num = 5000
test_num = 5000
train_csv_path = "train.csv"
val_csv_path = "val.csv"
test_csv_path = "test.csv"

create_labeled_dataset(ht, mt, train_num, val_num, test_num, train_csv_path, val_csv_path, test_csv_path)